# ಪಾಠ 13 - ಕೊಗ್ನೀ ಜ್ಞಾನ ಗ್ರಾಫ್‌ಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ಮೆಮೋರೀ


## ಕಾನ್ಫಿಗರ್ ಮಾಡಿ

ಈ ನೋಟ್ಬುಕ್ [**Cognee**](https://www.cognee.ai/) ಜ್ಞಾನ ಗೃಹಗಳು ಮತ್ತು **Microsoft Agent Framework** (MAF) ಬಳಸಿ ಸ್ಥಾಯಿ ನೆನಪಿನೊಂದಿಗೆ ಹೆಚ್ಚಿಸಬಹುದಾದ **ಕೋಡಿಂಗ್ ಸಹಾಯಕ** ಅನ್ನು ನಿರ್ಮಿಸುವ ವಿಧಾನವನ್ನು ತೋರಿಸುತ್ತದೆ.

Cognee ಅಸಂರಚಿತ ಪಠ್ಯವನ್ನು ವಕ್ಟರ್ ಎम्बೆಡ್ಡಿಂಗ್ಸ್‌ನಿಂದ ಬೆಂಬಲಿಸಲಾದ, ಸಂರಚಿತ, ಪ್ರಶ್ನಿಸುವ ಜ್ಞಾನ ಗ್ರಾಫ್ ಆಗಿ ಪರಿವರ್ತಿಸುತ್ತದೆ — ಇದು ನಿಮ್ಮ ಏಜೆಂಟ್‌ಗೆ ಶ್ರೀಮಂತ, ಸಂಬಂಧ-ಜ್ಞಾತ ದೀರ್ಘಕಾಲೀನ ನೆನಪನ್ನು ನೀಡುತ್ತದೆ.

### ನೀವು ಕಲಿಯುವುದು
1. **ಜ್ಞಾನ ಗ್ರಾಫ್ ನಿರ್ಮಿಸಿ**: ಡೆವಲಪರ್ ಪ್ರೊಫೈಲ್‌ಗಳು ಮತ್ತು ಉತ್ತಮ ಕ್ರಮಗಳನ್ನು ಸಂರಚಿತ, ಪ್ರಶ್ನಿಸಲು ಸಾಧ್ಯವಿರುವ ಜ್ಞಾನದಿಂದ ಪರಿವರ್ತಿಸಿ.
2. **Cognee ಅನ್ನು MAF ಜೊತೆಗೆ ಸಂಯೋಜಿಸಿ**: MAF ಏಜೆಂಟ್ Cognee ಜ್ಞಾನ ಗ್ರಾಫ್ ಅನ್ನು ಪ್ರಶ್ನಿಸಲು `@tool` ಫಂಕ್ಷನ್‌ಗಳನ್ನು ಬಳಸಿ.
3. **ಸೆಷನ್ ಜ್ಞಾನದ ಸಂವಾದಗಳು**: ಒಂದೇ ಸೆಷನ್ನಲ್ಲಿ ಬಹುತೇಕ ಪ್ರಶ್ನೆಗಳ ನಡುವಿನ ಸಂದರ್ಭದಲ್ಲಿ ನಿರ್ವಹಣೆ.
4. **ದೀರ್ಘಕಾಲೀನ ನೆನಪು**: ಪ್ರಮುಖ ಜ್ಞಾನಗಳನ್ನು ಸೆಷನ್‌ಗಳಾದ್ಯಾಂತ ಸ್ಥಾಯಿ ಮಾಡಿ ಹೊಸ ಸಂವಾದಗಳಲ್ಲಿ ಅದನ್ನು ಮತ್ತೆ ಪಡೆಯಿರಿ.

### ಅಗತ್ಯ ಉಳಿತಾಯಗಳು
- Python 3.9+
- ಸೆಷನ್ ನಿರ್ವಹಣೆಗೆ ರೆಡಿಸ್ ಸ್ಥಳೀಯವಾಗಿ ನಡಿಸಲಾಗುತ್ತಿದೆ (`docker run -d -p 6379:6379 redis`)
- LLM API ಕೀ (ಉದಾ., OpenAI) — `.env` ನಲ್ಲಿ `LLM_API_KEY` ಸೆಟ್ ಮಾಡಲ್ಪಡಿ
- `.env` ನಲ್ಲಿ `CACHING=true` (Cognee ಸೆಷನ್‌ಗಳಿಗೆ ಅಗತ್ಯ)
- ಚಾಟ್ ಮಾದರಿ ನಿಯೋಜಿಸಿರುವ Azure AI Foundry ಪ್ರಾಜೆಕ್ಟ್
- `.env` ನಲ್ಲಿ `AZURE_AI_PROJECT_ENDPOINT` ಮತ್ತು `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI ನಲ್ಲಿ ಪ್ರಾಮಾಣೀಕೃತ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ಏಜೆಂಟ್ ನೆನಪಿನ ವಿಧಗಳು

ಈ ನೋಟ್‌ಬುಕ್ ಮುಖ್ಯ ಪಾಠ 13 ನೋಟ್‌ಬುಕ್‌ನಿಂದ ಅದೇ ಮೂರು ನೆನಪುಗಳ ವಿಧಗಳನ್ನು ಪರಿಶೀಲಿಸುತ್ತದೆ, ಆದರೆ ದೀರ್ಘಕಾಲ ನೆನಪು ಬ್ಯಾಕ್ಎಂಡ್‌ಗಾಗಿ Cognee ಅನ್ನು ಬಳಸುತ್ತದೆ:

| ನೆನಪು ವಿಧ | ವಿಧಾನ | ಆಯુષ್ಯ |
|---|---|---|
| **ಕಾರ್ಯನಿರ್ವಹಣಾ** | `agent.create_session()` (MAF) | ಏಕ ಸಂಭಾಷಣೆ |
| **ಅಲ್ಪಾವಧಿ** | Cognee ಸೆಷನ್ ಕ್ಯಾಶೆ (Redis) | ಏಕ ಸೆಷನ್ |
| **ದೀರ್ಘಕಾಲಿಕ** | Cognee ಜ್ಞಾನ ಗ್ರಾಫ್ + ವೆಕ್ಟರ್‌ಗಳು | ಶಾಶ್ವತ |

### Cognee ಯ ನೆನಪು ವಾಸ್ತುಶಿಲ್ಪ
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## ಕೊಗ್ನೀ ಸಂಗ್ರಹಣೆಯನ್ನು ತಯಾರಿಸಿ


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## ಭಾಗ 1 — ಜ್ಞಾನ ಶಿಖರವನ್ನು ನಿರ್ಮಿಸುವುದು

ನಾವು ನಮ್ಮ ಕೋಡಿಂಗ್ ಸಹಾಯಕರಿಗಾಗಿ ಸಮಗ್ರ ಜ್ಞಾನ ಶಿಖರವನ್ನು ರಚಿಸಲು ಮೂರು ಪ್ರಕಾರದ ಡೇಟಾವನ್ನು ಇಂಜೆಸ್ಟ್ ಮಾಡುತ್ತೇವೆ:

1. **ಡೆವಲಪರ್ ಪ್ರೊಫೈಲ್** — ವೈಯಕ್ತಿಕ ಪರಿಣತಿ ಮತ್ತು ತಾಂತ್ರಿಕ ಹಿನ್ನೆಲೆ
2. **ಪೈಥನ್ ಉತ್ತಮ ಅಭ್ಯಾಸಗಳು** — ಪ್ರಾಯೋಗಿಕ ಮಾರ್ಗದರ್ಶಕರೊಂದಿಗೆ ಪೈಥನ್‌ನ ಜೆನ್
3. **ಐತಿಹಾಸಿಕ ಸಂಭಾಷಣೆಗಳು** — ಡೆವಲಪರ್‌ಗಳು ಮತ್ತು AI ಸಹಾಯಕರ ನಡುವಿನ ಹಿಂದಿನ ಪ್ರಶ್ನೋತ್ತರ ಸೆಷನ್‌ಗಳು


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ಜ್ಞಾನ ಗ್ರಾಫ್ ವೀಕ್ಷಣೆ

Cognee ತೆಗೆದುಕೊಂಡ ಅಂಶಗಳು ಮತ್ತು ಸಂಬಂಧಗಳ ಸಂವಹನಾತ್ಮಕ HTML ವೀಕ್ಷಣೆಯನ್ನು ರೆಂಡರ್ ಮಾಡಬಹುದು.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify ನೊಂದಿಗೆ ಸ್ಮರಣೆಯನ್ನು ಸುದ್ಧಿಗೊಳಿಸಿ

`memify()` ಜ್ಞಾನದ ಗ್ರಾಫ್ ಅನ್ನು ವಿಶ್ಲೇಷಿಸಿ ಬುದ್ಧಿವಂತ ನಿಯಮಗಳನ್ನು ರಚಿಸುತ್ತದೆ — ಮಾದರಿಗಳು, ಉತ್ತಮ ಅಭ್ಯಾಸಗಳು ಮತ್ತು ಸಿದ್ಧಾಂತಗಳ ನಡುವೆ ಸಂಬಂಧಗಳನ್ನು ಗುರುತಿಸುವುದು.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## ಭಾಗ 2 — Cognee ಟೂಲ್ಸ್‌ನೊಂದಿಗೆ MAF ಏಜೆಂಟ್

ಈಗ ನಾವು Cognee ನ ಜ্ঞান ಗ್ರಾಫ್ ಅನ್ನು `@tool` ಫಂಕ್ಷನ್ಗಳ ಮೂಲಕ ಪ್ರಶ್ನಿಸುವ MAF ಏಜೆಂಟ್ ಅನ್ನು ಸೃಷ್ಟಿಸುತ್ತೇವೆ. ಈ ಮೂಲಕ ಏಜೆಂಟ್ ಸಂವಾದಾತ್ಮಕ ಹಿನ್ನೆಲೆಯಲ್ಲಿ ಸೆಷನ್ಗಳ ಮೂಲಕ ನಿರ್ವಹಿಸುವಾಗ ಗ್ರಾಫ್-ಜಾಗೃತಿ_semantic_ ಹುಡುಕಾಟದ ಪೂರ್ಣ ಶಕ್ತಿಯನ್ನು ಉಪಯೋಗಿಸಬಹುದು.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## ಸೆಷನ್‌ಗಳೊಂದಿಗೆ ಕಾರ್ಯಿಸುವ ಸ್ಮೃತಿ

`AgentSession` (`agent.create_session()` ಮೂಲಕ ರಚಿಸಲಾಗಿದೆ) ಸೆಷನ್‌ನ ಒಳಗೆ ಕಾರ್ಯಿಸುವ ಸ್ಮೃತಿಯನ್ನು ಒದಗಿಸುತ್ತದೆ. ಏಜೆಂಟ್ ಹಿಂದಿನ ಸಂದೇಶಗಳನ್ನು ಉಲ್ಲೇಖಿಸಬಹುದು ಹಾಗೆಯೇ Cognee ರ ದೀರ್ಘಕಾಲೀನ ಜ್ಞಾನ ಗ್ರಾಫ್ ಅನ್ನು ಪ್ರಶ್ನಿಸಬಹುದು.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## ಹೊಸ ಸೆಷನ್ — ದೀರ್ಘಕಾಲದ ಸ್ಮರಣೆ ಉಳಿಯುತ್ತದೆ

ಹೊಸ ಸೆಷನನ್ನು ಪ್ರಾರಂಭಿಸುವುದು ವರ್ಕಿಂಗ್ ಮೆಮೊರಿಯನ್ನು ಕ್ಲೀರ್ ಮಾಡುತ್ತದೆ, ಆದರೆ Cognee ಜ್ಞಾನ ಗ್ರಾಫ್ ಇನ್ನೂ ಲಭ್ಯವಿದೆ. ಏಜೆಂಟ್ ಸಂಪೂರ್ಣವಾಗಿ ಹೊಸ ಸಂಭಾಷಣೆಯಲ್ಲಿ ಅದೇ ದೀರ್ಘಕಾಲದ ಜ್ಞಾನವನ್ನು ಪಡೆಯಬಹುದು.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## ಸಾರಾಂಶ

ಈ ನೋಟಬುಕ್‌ನಲ್ಲಿ ನೀವು **MAF ನ ಕಾರ್ಯಿಸುವ ಸ್ಮರಣೆ** (`agent.create_session()`) ಮತ್ತು **Cognee ನ ದೀರ್ಘಕಾಲीन ಜ್ಞಾನ ಗ್ರಾಫ್** ಅನ್ನು ಸಂಯೋಜಿಸುವ ಕೋಡಿಂಗ್ ಸಹಾಯಕನನ್ನು ನಿರ್ಮಿಸಿದ್ದೀರಿ.

### ನೀವು ಕಲಿತದ್ದು
1. **ಜ್ಞಾನ גרಾಫ್ ನಿರ್ಮಾಣ**: Cognee ಅಸಂರಚಿತ ಪಠ್ಯವನ್ನು ಆಹರಿಸಿ ಗ್ರಾಫ್ + ವೆಕ್ಟರ್ ನೆನಪನ್ನು ನಿರ್ಮಿಸುತ್ತದೆ.
2. **ಮೆಮಿಫೈ ಮೂಲಕ ಗ್ರಾಫ್ ಅನ್ನು ಸಮೃದ್ಧಗೊಳನೆ**: ನಿಮ್ಮ ಇರುವ ಗ್ರಾಫ್ ಮೇಲೆ ಪಡೆಯಲಾದ ಸತ್ಯಗಳ ಮತ್ತು ಶ್ರೀಮಂತ ಸಂಬಂಧಗಳನ್ನು ಹೊಂದಿಸಲಾಗಿದೆ.
3. **MAF + Cognee ಒಂದುಗೂಡಿಕೆ**: `@tool` ಫಂಕ್ಷನ್ಸ್ MAF ಏಜೆಂಟ್ Cognee ಗ್ರಾಫ್ ನೊಂದಿಗೆ ಸಹಜವಾಗಿ ಪ್ರಶ್ನೆಮಾಡಲು ಅವಕಾಶ ನೀಡುತ್ತವೆ.
4. **ಕಾರ್ಯಾಚರಣೆ ಸ್ಮರಣೆ + ದೀರ್ಘಕಾಲೀನ ಸ್ಮರಣೆ**: `AgentSession` (`agent.create_session()` ಮೂಲಕ) ಸೆಷನ್ ಸಂದರ್ಭವನ್ನು ಒದಗಿಸುತ್ತದೆ ಆಗಾಗ Cognee ಸ್ಥಾಯಿ ಜ್ಞಾನ ಒದಗಿಸುತ್ತದೆ.
5. **NodeSets ಜೊತೆ ಫಿಲ್ಟರ್ ಮಾಡಿದ ಹುಡುಕಾಟ**: ಜ್ಞಾನ ಗ್ರಾಫ್ ನ ನಿರ್ದಿಷ್ಟ ಉಪಸಂಕಲನವನ್ನು ಗುರಿಯಾಗಿಸುವುದು (ಉದಾ: ಕೇವಲ ತತ್ವಗಳು).

### ಪ್ರಮುಖ ಕಲಿಕೆಗಳು
- **Cognee** ಕಚ್ಛಾ ಪಠ್ಯವನ್ನು ರಚನಾತ್ಮಕ, ಸಂಬಂಧ-ಜಾಗೃತಿ ನೆನಪಾಗಿಸುವ ಮೂಲಕ ಪರಿವರ್ತಿಸುತ್ತದೆ — ಸಮತಲ ವೆಕ್ಟರ್ ಸ್ಟೋರ್ ಕ್ಕಿಂತ ಬಲವಾದದು.
- **`@tool` ಫಂಕ್ಷನ್ಸ್** MAF ಏಜೆಂಟ್ ಗಳನ್ನು ಬಾಹ್ಯ ಜ್ಞಾನ ವ್ಯವಸ್ಥೆಗಳೊಂದಿಗೆ ಸ್ಫಟಿಕವಾಗಿ ಸೇರ್ಸುತ್ತವೆ.
- **`AgentSession`** (`agent.create_session()` ಮೂಲಕ) ಪ್ರತಿಸಂವಾದದ ಸಂದರ್ಭವನ್ನು ದೀರ್ಘಾವಧಿ ಜ್ಞಾನದಿಂದ ವಿಭಿನ್ನವಾಗಿ ಉಳಿಸುತ್ತದೆ.
- ಆದೇ ಜ್ಞಾನ ಗ್ರಾಫ್ ಅನೇಕ ಸೆಷನ್ ಮತ್ತು ಏಜೆಂಟ್ ಗಳು ಬಳಸಿಕೊಳ್ಳಬಹುದು.

### ವಾಸ್ತವಿಕ ಜಾಗತಿಕ ಅನ್ವಯಿಕೆಗಳು
- **ಡೆವಲಪರ್ ಕೋಪೈಲಟ್ ಗಳು**: ಕೋಡ್ ಪರಿಶೀಲನೆ, ಘಟನಾ ವಿಶ್ಲೇಷಣೆ, ವಾಸ್ತುಕಲ ಸಹಾಯಕರಾಗಿ
- **ಗ್ರಾಹಕ ಮುಖಾಮುಖಿ ಕೋಪೈಲಟ್ ಗಳು**: ಉತ್ಪನ್ನ ದಾಖಲೆಗಳು,FAQ ಗಳು ಮತ್ತು CRM ಟಿಪ್ಪಣಿಗಳ ಮೇಲೆ ಬೆಂಬಲ ಏಜೆಂಟ್ ಗಳು
- **ಅಂತರಂಗ ತಜ್ಞ ಕೋಪೈಲಟ್ ಗಳು**: ನೀತಿ, ಕಾನೂನು ಅಥವಾ ಸುರಕ್ಷತಾ ಸಹಾಯಕರಾಗಿ ಮಾರ್ಗಸೂಚಿಗಳ ಮೇಲೆ ತರ್ಕಿಸುವವರು
- **ಏಕೀಕೃತ ಡೇಟಾ ಪದರಗಳು**: ರಚನಾತ್ಮಕ ಮತ್ತು ಅಸಂರಚಿತ ಡೇಟಾವನ್ನು ಒಂದೇ ಪ್ರಶ್ನೆಯ ಸಾಧ್ಯವಾದ ಗ್ರಾಫ್ ನಲ್ಲಿ ಸಂಯೋಜನೆ

### ಮುಂದಿನ ಹಂತಗಳು
- Cognee ನಲ್ಲಿ ಕಾಲಾತೀತ ಜಾಗೃತಿ ಜೊತೆ ಪ್ರಯೋಗಮಾಡಿ
- ಕ್ಷೇತ್ರ-ನಿರ್ದಿಷ್ಟ ಗ್ರಾಫ್ ಗುಣಮಟ್ಟಕ್ಕಾಗಿ OWL ಓಂಟಾಲಜಿ ವ್ಯಾಖ್ಯಾನಿಸಿ
- ಸಮಯದೊಂದಿಗೆ ವಾಪಸ್ಸು ಪಡೆಯುವಿಕೆಯನ್ನು ಉತ್ತಮಗೊಳಿಸಲು ಬಳಕೆದಾರ ಪ್ರತಿಕ್ರಿಯಾ ಲೂಪ್ಗಳನ್ನು ಸೇರಿಸಿ
- ಒಂದೇ Cognee ಸ್ಮರಣೆ ಪದರವನ್ನು ಹಂಚಿಕೊಳ್ಳುವ ಬಹು-ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳೆಡೆಗೆ ವಿಸ್ತರಿಸಿ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಸೂಚನೆ**:  
ಈ ದಾಖಲೆಯನ್ನು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಗೆ ಪ್ರಯತ್ನಿಸುವುದಾದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸತ್ಯತೆಗಳು ಇರಬಹುದಾಗಿದೆ ಎಂಬುದನ್ನು ದಯವಿಟ್ಟು ಗಮನದಲ್ಲಿರಿಸಿ. ಮೂಲದ ಭಾಷೆಯಲ್ಲಿ ಇರುವ ಮೂಲ ದಾಖಲೆ ಆದೇಶಾತ್ಮಕ ಮೂಲವಾಗಿರುತ್ತದೆ. ಮಹತ್ವದ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೆ ಭಾಷೆಯ ಅರ್ಥಮೈಮರೆಯಿಗೂ ಅಥವಾ ತಪ್ಪು ಅರ್ಥಗರ್ಭಿತ್ತಿಗೆ ನಾವು ಹೊಣೆಗಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
